# Clase 054 — Proyecto end-to-end: visión, datos, exploración, preparación

Primera mitad de un proyecto real: framing, EDA, split estratificado sin contaminarse y un
`Pipeline` + `ColumnTransformer` que limpia, encodea y escala sin data leakage. Usamos un
dataset sintético tipo "housing" (numéricas + categórica + NaN) para correr offline.

Requiere: `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

np.random.seed(42)
rng = np.random.default_rng(42)

## 1. Framing + dataset sintético "housing"

Tarea: **regresión** del valor de la vivienda; métrica: **RMSE en dólares**; baseline:
`LinearRegression`. Construimos un DataFrame con una categórica (`ocean_proximity`) y NaN
reales en `total_bedrooms`.

In [ ]:
n = 4000
median_income = rng.gamma(2.0, 2.0, n)
households = rng.integers(80, 3000, n).astype(float)
total_rooms = households * rng.uniform(3, 8, n)
total_bedrooms = total_rooms * rng.uniform(0.15, 0.30, n)
prox = rng.choice(['INLAND', 'NEAR_BAY', 'NEAR_OCEAN', '<1H_OCEAN'], n,
                  p=[0.35, 0.2, 0.2, 0.25])
prox_bonus = pd.Series(prox).map({'INLAND': 0, 'NEAR_BAY': 60, 'NEAR_OCEAN': 80,
                                  '<1H_OCEAN': 40}).values
value = (40_000 + median_income * 45_000 + prox_bonus * 1_000
         + rng.normal(0, 25_000, n)).clip(20_000, 500_001)

df = pd.DataFrame({'median_income': median_income, 'households': households,
                   'total_rooms': total_rooms, 'total_bedrooms': total_bedrooms,
                   'ocean_proximity': prox, 'median_house_value': value})
df.loc[rng.choice(n, 150, replace=False), 'total_bedrooms'] = np.nan  # NaN reales
print(df.shape, '| NaN en total_bedrooms:', int(df.total_bedrooms.isna().sum()))
print(df.describe().round(1).T[['mean', 'min', 'max']])

## 2. EDA mínimo: distribuciones e info

`describe`/`hist` para conocer el dato antes de modelar.

In [ ]:
print(df['ocean_proximity'].value_counts().to_string())
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, ['median_income', 'total_rooms', 'median_house_value']):
    ax.hist(df[col].dropna(), bins=40, color='#37a')
    ax.set_title(col)
plt.tight_layout(); plt.show()

## 3. Split estratificado por income bucket

Un split aleatorio ingenuo puede sesgar el test. Estratificamos por un bucket de
`median_income` para que train y test tengan la misma composición.

In [ ]:
df['income_cat'] = pd.cut(df['median_income'], bins=[0, 1.5, 3, 4.5, 6, np.inf],
                          labels=[1, 2, 3, 4, 5])
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(sss.split(df, df['income_cat']))
strat_train, strat_test = df.iloc[tr_idx], df.iloc[te_idx]

comp = pd.DataFrame({
    'train': strat_train['income_cat'].value_counts(normalize=True).sort_index(),
    'test':  strat_test['income_cat'].value_counts(normalize=True).sort_index()})
print(comp.round(3))
assert (comp['train'] - comp['test']).abs().max() < 0.02, 'proporciones deben coincidir'
print('OK: distribucion de income_cat casi identica en train y test')

## 4. Feature engineering + ColumnTransformer sin leakage

Derivamos `rooms_per_household` y `bedrooms_per_room`, y armamos el `ColumnTransformer`:
numéricas → imputer(median) + scaler; categórica → one-hot. Todo se fitea SOLO en train.

In [ ]:
def add_features(d):
    d = d.copy()
    d['rooms_per_household'] = d['total_rooms'] / d['households']
    d['bedrooms_per_room'] = d['total_bedrooms'] / d['total_rooms']
    return d

drop_cols = ['median_house_value', 'income_cat']
X_train = add_features(strat_train).drop(columns=drop_cols)
y_train = strat_train['median_house_value'].values
X_test = add_features(strat_test).drop(columns=drop_cols)
y_test = strat_test['median_house_value'].values

num_cols = X_train.select_dtypes('number').columns.tolist()
cat_cols = ['ocean_proximity']
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
print('numericas:', num_cols)
print('categoricas:', cat_cols)

## 5. Baseline LinearRegression: RMSE train vs test

`fit` solo en train; el test se toca una única vez para el scoring final.

In [ ]:
model = Pipeline([('pre', pre), ('lr', LinearRegression())])
model.fit(X_train, y_train)
rmse_tr = root_mean_squared_error(y_train, model.predict(X_train))
rmse_te = root_mean_squared_error(y_test, model.predict(X_test))
print(f'RMSE train: {rmse_tr:,.0f} USD')
print(f'RMSE test : {rmse_te:,.0f} USD')
print('sin warnings de leakage: todo el fit ocurrio dentro del Pipeline sobre train')

## 6. Visual: predicho vs real en test

In [ ]:
pred_te = model.predict(X_test)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test, pred_te, s=8, alpha=0.4, color='#37a')
lims = [y_test.min(), y_test.max()]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlabel('valor real'); ax.set_ylabel('valor predicho')
ax.set_title('Baseline lineal: predicho vs real (test)')
plt.tight_layout(); plt.show()

## Ejercicios

1. Reemplazá `SimpleImputer(median)` por `KNNImputer(n_neighbors=5)` en las numéricas y compará
   el RMSE de test. ¿Cuál ganó y en cuánto?
2. Agregá una feature `population_per_household` (inventala a partir de columnas existentes) y
   medí si el RMSE mejora. ¿Es feature engineering con palanca real?
3. Provocá el error `Found unknown categories`: creá en test una categoría de
   `ocean_proximity` inexistente en train y mostrá cómo `handle_unknown='ignore'` lo resuelve.
4. Compará `StandardScaler` vs sin escalar sobre `Ridge(alpha=10)`. ¿Por qué el escalado cambia
   los coeficientes en un modelo regularizado?

## Conclusiones

- Framear primero (tarea, métrica, baseline) evita "cualquier modelo anda".
- El split estratificado por income bucket da un test representativo, no sesgado por azar.
- `ColumnTransformer` + `Pipeline` encadena limpieza, encoding y scaling sin leakage.
- Regla de oro: todo `.fit()` (imputers, scalers, encoders) ocurre solo en train.